# Day 29 — Random Forest Classifier

**Veda Technology · Level 2 · Day 29 · Data Science Track**

Build a Random Forest classification model and compare it with a single Decision Tree.

## Objective
Understand ensemble learning and why combining multiple trees can improve generalization.

## Tools
Python, Pandas, Scikit-learn, Jupyter Notebook, Matplotlib

## Dataset
Breast Cancer Wisconsin Dataset from scikit-learn.


## 1. Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 2. Load and Inspect Dataset

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")
print("Shape:", X.shape)
print("Target names:", data.target_names)
print("Missing values:", X.isnull().sum().sum())
X.head()


## 3. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 4. Decision Tree Baseline

In [ ]:
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)
tree_train = accuracy_score(y_train, tree.predict(X_train))
tree_test = accuracy_score(y_test, tree.predict(X_test))
print(f"Decision Tree training accuracy: {tree_train:.4f}")
print(f"Decision Tree testing accuracy:  {tree_test:.4f}")


## 5. Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_train = accuracy_score(y_train, rf.predict(X_train))
rf_test = accuracy_score(y_test, rf.predict(X_test))
print(f"Random Forest training accuracy: {rf_train:.4f}")
print(f"Random Forest testing accuracy:  {rf_test:.4f}")


## 6. Performance Comparison

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest"],
    "Training Accuracy": [tree_train, rf_train],
    "Testing Accuracy": [tree_test, rf_test]
})
comparison.round(4)


In [ ]:
plt.figure(figsize=(8,5))
x=range(len(comparison))
w=0.35
plt.bar([i-w/2 for i in x], comparison["Training Accuracy"], width=w, label="Training")
plt.bar([i+w/2 for i in x], comparison["Testing Accuracy"], width=w, label="Testing")
plt.xticks(list(x), comparison["Model"])
plt.ylabel("Accuracy")
plt.title("Decision Tree vs Random Forest")
plt.ylim(0,1.05)
plt.legend()
plt.tight_layout()
plt.show()


## 7. Experiment with `n_estimators`

In [ ]:
estimators = [10, 25, 50, 100, 200, 300]
rows = []
for n in estimators:
    model = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    rows.append({
        "n_estimators": n,
        "train_accuracy": accuracy_score(y_train, model.predict(X_train)),
        "test_accuracy": accuracy_score(y_test, model.predict(X_test))
    })
estimator_results = pd.DataFrame(rows)
estimator_results.round(4)


In [ ]:
plt.figure(figsize=(9,5))
plt.plot(estimator_results["n_estimators"], estimator_results["train_accuracy"], marker="o", label="Training")
plt.plot(estimator_results["n_estimators"], estimator_results["test_accuracy"], marker="o", label="Testing")
plt.xlabel("Number of Trees")
plt.ylabel("Accuracy")
plt.title("Random Forest Performance vs n_estimators")
plt.grid(alpha=.3)
plt.legend()
plt.tight_layout()
plt.show()


## 8. Best Tested Number of Trees

In [ ]:
best_test = estimator_results["test_accuracy"].max()
best_n = int(estimator_results.loc[estimator_results["test_accuracy"] == best_test, "n_estimators"].min())
print("Best tested n_estimators:", best_n)
print(f"Best testing accuracy: {best_test:.4f}")


## 9. Feature Importance

In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)
importance.head(15)


In [ ]:
top = importance.head(15).sort_values("Importance")
plt.figure(figsize=(9,7))
plt.barh(top["Feature"], top["Importance"])
plt.xlabel("Importance")
plt.title("Top 15 Random Forest Feature Importances")
plt.tight_layout()
plt.show()


## 10. Final Evaluation and Overfitting Analysis

The training-testing accuracy gap is:

`Accuracy Gap = Training Accuracy − Testing Accuracy`

A smaller gap can indicate better generalization, but it should be considered together with the actual test performance.


In [ ]:
comparison["Accuracy Gap"] = comparison["Training Accuracy"] - comparison["Testing Accuracy"]
comparison.round(4)

print("Classification Report:")
print(classification_report(y_test, rf.predict(X_test), target_names=data.target_names))
print("Confusion Matrix:")
print(confusion_matrix(y_test, rf.predict(X_test)))


## 11. Interview Questions

### What is a Random Forest?
An ensemble learning algorithm that combines many decision trees.

### Why does Random Forest usually generalize better than one Decision Tree?
A single tree can overfit. Combining many trees generally reduces variance and makes predictions more robust.

### What is ensemble learning?
Combining multiple models to produce a stronger overall prediction.

### What does `n_estimators` control?
The number of trees in the Random Forest.

### What is `feature_importances_`?
An estimate of how much each feature contributes to impurity reduction across the forest.

## Conclusion
This project demonstrates Random Forest classification, comparison with a Decision Tree, `n_estimators` experimentation, feature importance, and generalization analysis.
